# Embeddings of a pretrained transformer with cosine similarity

Embeds both texts of every pair with a pretrained transformer WITHOUT fine-tuning and decides
by the cosine similarity of the two vectors whether they were written by the same author. One
forward pass per text gives two vectors - the first token (CLS) and the mean of all tokens -
and each gives its own result row. The threshold of each is selected on the validation pairs
from every similarity observed there and applied once to the test pairs.

A restarted run continues where it stopped - the similarities are saved in chunks, a finished
run is not repeated and chunks of other settings or other data stop the run.

In [ ]:
SMOKE = False  # True - a few rows of every data set, a chunk of 2 pairs
# label in the file names of a repeated run, so two runs never share a file
RUN_TAG = ""
# Hugging Face id, or the folder of a saved model
MODEL_ID = "allenai/longformer-base-4096"
DEVICE = "cuda"  # "cpu" only for the CPU checks - there is no fallback to the CPU

Standard libraries

In [ ]:
import functools
import json
import logging
import re
import shutil
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

Data and plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

Torch and Hugging Face

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, AutoTokenizer, PreTrainedTokenizerBase

The shared modules of the thesis

In [ ]:
from avcore import data, evaluation, runs, tracking, transformer_training

# Configuration

The model - the notebook runs for these three models only, named after the end of MODEL_ID

In [ ]:
MODEL_NAMES = ["longformer-base-4096", "bigbird-roberta-base", "ModernBERT-large"]
MODEL_NAME = Path(MODEL_ID).name
if MODEL_NAME not in MODEL_NAMES:
    raise ValueError(f"MODEL_ID {MODEL_ID!r} does not end with one of {MODEL_NAMES}.")
POOLINGS = ["cls_embedding", "mean_pooling_embedding"]
CLASSIFIER = "cosine_similarity"
DEVELOPER_INITIALS = "JP"
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Sizes - the token limit of a text (the same as the classification heads), the chunk size

In [ ]:
MAX_LENGTH = transformer_training.MAX_LENGTH
CHUNK_PAIRS = 500
SMOKE_ROWS = {"validation": 8, "test": 12}
if SMOKE:
    CHUNK_PAIRS = 2

The data files - the training pairs are not needed, nothing is trained

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
VALIDATION_PATH = DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE
TEST_PATH = DATA_DIR / "processed" / data.TEST_CLEANED_FILE
DATA_PATHS = [VALIDATION_PATH, TEST_PATH]